In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
data = pd.read_csv('input_data/DepMap/CRISPRGeneEffect.csv')
data

In [ ]:
data = data.rename(columns = {'Unnamed: 0':'sample'})
data

In [ ]:
data = data.set_index(data['sample'])
data = data.drop(columns = ['sample'])
data

In [ ]:
# Function to extract numbers within parentheses
def extract_number(col_name):
    match = re.search(r'\((\d+)\)', col_name)
    return match.group(1) if match else col_name  # Keep original name if no match

# Rename columns
data.columns = [extract_number(col) for col in data.columns]
data

In [ ]:
corr = data.corr(method = 'pearson')
corr

In [ ]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
query_biomarker = main_csv[['entrez_id_query', 'entrez_id_biomarker']]

# Convert entrez IDs to float for consistency with correlation matrix
query_biomarker['entrez_id_query'] = query_biomarker['entrez_id_query'].astype(float)
query_biomarker['entrez_id_biomarker'] = query_biomarker['entrez_id_biomarker'].astype(float)

# Ensure correlation matrix index/columns are also float
corr.index = corr.index.astype(float)
corr.columns = corr.columns.astype(float)

data.columns = data.columns.astype(float)

# 🔹 Calculate coessentiality variance of each gene
coessentiality_variance = data.var(axis=0)
essential_avg = data.mean(axis=0)

# 🔹 Function to get coessentiality correlation
def get_correlation(row):
    if pd.isna(row['entrez_id_query']) or pd.isna(row['entrez_id_biomarker']):
        return 0.0
    return corr.loc[row['entrez_id_query'], row['entrez_id_biomarker']] if (
        row['entrez_id_query'] in corr.index and row['entrez_id_biomarker'] in corr.columns
    ) else 0.0

# 🔹 Function to get query gene's coessentiality variance
def get_query_variance(row):
    gene_id = row['entrez_id_query']
    if pd.isna(gene_id):
        return np.nan
    return coessentiality_variance.get(gene_id, np.nan)

def get_query_essentiality(row):
    gene_id = row['entrez_id_query']
    if pd.isna(gene_id):
        return np.nan
    return essential_avg.get(gene_id, np.nan)

# 🔹 Apply both
query_biomarker['Correlation'] = query_biomarker.apply(get_correlation, axis=1)
query_biomarker['Biomarker_Essentiality_Variance'] = query_biomarker.apply(get_query_variance, axis=1)
query_biomarker['Biomarker_Essentiality_Average'] = query_biomarker.apply(get_query_essentiality, axis=1)
query_biomarker['Biomarker_Essentiality_Average'] = query_biomarker['Biomarker_Essentiality_Average']* -1

query_biomarker.to_csv('input_data/3_ML_outputs/feature_output/coessentiality_biomarker_query.csv',index=False)
query_biomarker


In [ ]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
query_target1 = main_csv[['entrez_id_query', 'entrez_id_target1']]

query_target1['entrez_id_query'] = query_target1['entrez_id_query'].astype(float)
query_target1['entrez_id_target1'] = query_target1['entrez_id_target1'].astype(float)

corr.index = corr.index.astype(float)
corr.columns = corr.columns.astype(float)


def get_correlation_target1(row):
    if pd.isna(row['entrez_id_query']) or pd.isna(row['entrez_id_target1']):
        return 0.0
    return corr.loc[row['entrez_id_query'], row['entrez_id_target1']] if (
        row['entrez_id_query'] in corr.index and row['entrez_id_target1'] in corr.columns
    ) else 0.0
query_target1['Correlation'] = query_target1.apply(get_correlation_target1, axis=1)

query_target1.to_csv('input_data/3_ML_outputs/feature_output/coessentiality_target1_query.csv',index=False)
query_target1


In [ ]:
query_target2 = main_csv[['entrez_id_query', 'entrez_id_target2']]

query_target2['entrez_id_query'] = query_target2['entrez_id_query'].astype(float)
query_target2['entrez_id_target2'] = query_target2['entrez_id_target2'].astype(float)

corr.index = corr.index.astype(float)
corr.columns = corr.columns.astype(float)


# Reuse or re-calculate variance if not already done
coessentiality_variance = data.var(axis=0)
essential_avg = data.mean(axis=0)

# 🔹 Function to get correlation between query and target2
def get_correlation_target2(row):
    if pd.isna(row['entrez_id_query']) or pd.isna(row['entrez_id_target2']):
        return 0.0
    return corr.loc[row['entrez_id_query'], row['entrez_id_target2']] if (
        row['entrez_id_query'] in corr.index and row['entrez_id_target2'] in corr.columns
    ) else 0.0

# Apply both
query_target2['Correlation'] = query_target2.apply(get_correlation_target2, axis=1)

query_target2.to_csv('input_data/3_ML_outputs/feature_output/coessentiality_target2_query.csv',index=False)
query_target2


In [ ]:
import pandas as pd

def essentiality_percentage_cell_line(df, threshold=-0.6):
    result = {}
    for col in df.columns:
        # only process numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            total = df[col].notna().sum()   # exclude NaN from denominator
            if total > 0:
                count = (df[col] < threshold).sum()
                result[col] = (count / total) * 100
            else:
                result[col] = None
    # return as DataFrame
    return pd.DataFrame(list(result.items()), columns=["entrez_id", "Essentiality_Percentage"])



essentiality_percentage = essentiality_percentage_cell_line(data, threshold=-0.6)
essentiality_percentage

In [ ]:
essentiality_percentage.to_csv('input_data/3_ML_outputs/feature_output/essentiality_percentage_per_gene.csv',index=False)